# Amazon Beauty Product Reviews

The cleaning rules are stored in `cleaning_script.py`. The notebook loads the original data, calls the cleaning function from the `.py` file, and documents the verification results.

## 1. Load the original source

The source is a 4,687-row sample of Amazon Beauty reviews. The original dataframe is kept unchanged in `df_original`; the cleaning function works on a copy.

In [17]:
import pandas as pd
from cleaning_script import load_source_data, clean_data, validate_data, make_record_checks, save_outputs

df_original = load_source_data()
print("Rows:", len(df_original))
print("Columns:", len(df_original.columns))

Rows: 4687
Columns: 14


## 3. Initial inspection

I inspected the columns, data types, missing values, duplicates, ranges, and text formatting before applying the cleaning rules.

In [18]:
df_original.info()

print("\nMissing values by column:")
print(df_original.isna().sum())

print("\nExact duplicates:", df_original.duplicated().sum())
print("Rating values:", sorted(df_original["rating"].unique()))
print("Verified purchase counts:")
print(df_original["verified_purchase"].value_counts())
print("Helpful vote range:", df_original["helpful_vote"].min(), "to", df_original["helpful_vote"].max())
print("Negative helpful votes:", (df_original["helpful_vote"] < 0).sum())
print("Negative prices:", (df_original["product_price"] < 0).sum())
print("HTML <br> reviews:", df_original["review_text"].str.contains("<br", regex=False, na=False).sum())
converted = pd.to_datetime(df_original["timestamp"], unit="ms", errors="coerce")
print("Timestamp conversion failures:", converted.isna().sum())
print("Earliest timestamp:", converted.min())
print("Latest timestamp:", converted.max())

<class 'pandas.DataFrame'>
RangeIndex: 4687 entries, 0 to 4686
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   user_id              4687 non-null   str    
 1   parent_asin          4687 non-null   str    
 2   rating               4687 non-null   float64
 3   review_title         4687 non-null   str    
 4   review_text          4687 non-null   str    
 5   timestamp            4687 non-null   int64  
 6   helpful_vote         4687 non-null   int64  
 7   verified_purchase    4687 non-null   int64  
 8   product_title        4687 non-null   str    
 9   product_description  4687 non-null   str    
 10  product_price        4687 non-null   float64
 11  product_brand        4687 non-null   str    
 12  product_categories   4687 non-null   str    
 13  image_url            4687 non-null   str    
dtypes: float64(2), int64(3), str(9)
memory usage: 4.2 MB

Missing values by column:
user_id            

## 4. Apply cleaning rules from `cleaning_script.py`


In [19]:
df = clean_data(df_original)
print("Starting rows:", len(df_original))
print("Final rows:", len(df))
print("Rows removed:", len(df_original) - len(df))
print("Columns after cleaning:", len(df.columns))
df.head()

Starting rows: 4687
Final rows: 4565
Rows removed: 122
Columns after cleaning: 15


,user_id,parent_asin,rating,review_title,review_text,timestamp,helpful_vote,verified_purchase,product_title,product_description,product_price,product_brand,product_categories,image_url,review_length
0,AFSKPY37N3C43SOI5IEXEK5JSIYA,B08P2DZB4X,5.0,Great for at home use and so easy to use!,This is perfect for my between salon visits. I...,2021-07-27 13:04:04.559,0,False,NIRA Skincare Laser & Serum Bundle - Includes ...,,15.95,,,https://m.media-amazon.com/images/I/31OE9icA9x...,388
1,AFSKPY37N3C43SOI5IEXEK5JSIYA,B07RBSLNFR,5.0,A little goes a long way!,This is a really nice moisturizing lotion. It ...,2021-05-16 17:00:30.697,0,False,OGANA CELL Peptide Concentrating Amazing Lotio...,,15.95,,,https://m.media-amazon.com/images/I/21R88ZimSj...,351
2,AFSKPY37N3C43SOI5IEXEK5JSIYA,B07SLFWZKN,3.0,Just ok,I try to get Keratin treatments every 3 months...,2021-04-29 23:05:01.209,0,False,Keratin Secrets Do It Yourself Home Keratin Sy...,,49.95,,,https://m.media-amazon.com/images/I/5163R0SItj...,764
3,AFSKPY37N3C43SOI5IEXEK5JSIYA,B08JTNQFZY,5.0,Good quality hair brush!,"Really nice small brush. Made well, nice wood ...",2021-04-08 17:50:19.785,0,False,GAINWELL,,15.95,,,https://m.media-amazon.com/images/I/41HKskjrzV...,280
4,AFSKPY37N3C43SOI5IEXEK5JSIYA,B08GLG6W8T,5.0,Great refreshing skin care routine!,I had never tried anything for my skin consist...,2021-02-14 16:13:56.253,0,False,Hanhoo Red pomegranate 6pcs Skin Care Set (6pc...,,15.95,,,https://m.media-amazon.com/images/I/51dY7GP0gV...,726


## 5. Verification


In [20]:
validation = validate_data(df_original, df)

print("\n1. DATASET SIZE")
print("Original rows:", validation["starting_rows"])
print("Original columns:", validation["starting_columns"])
print("Final rows:", validation["final_rows"])

print("\n2. MISSING VALUES IN ORIGINAL DATA")
print(df_original.isna().sum())

print("\n3. EXACT DUPLICATES")
print("Exact duplicate rows:", validation["exact_duplicates"])

print("\n4. DATA TYPES")
print(df_original.dtypes)

print("\n5. RATING CHECK")
print("Rating values:", sorted(df_original["rating"].unique()))
print("Invalid ratings:", validation["invalid_ratings"])

print("\n6. VERIFIED PURCHASE")
print("Original counts:")
print(df_original["verified_purchase"].value_counts())
print("Cleaned counts:")
print(df["verified_purchase"].value_counts())

print("\n7. HELPFUL VOTES")
print("Minimum:", df_original["helpful_vote"].min())
print("Maximum:", df_original["helpful_vote"].max())
print("Negative values:", validation["negative_helpful_votes"])

print("\n8. TIMESTAMP CHECK")
converted = pd.to_datetime(df_original["timestamp"], unit="ms", errors="coerce")
print("Failed conversions:", validation["timestamp_conversion_failures"])
print("Earliest:", converted.min())
print("Latest:", converted.max())

print("\n9. PRICE CHECK")
print("Negative prices:", validation["negative_prices"])

print("\n10. TEXT CHECK")
print("Reviews with HTML line breaks in original:", validation["html_breaks_original"])
print("HTML line breaks remaining after cleaning:", df["review_text"].str.contains("<br", regex=False, na=False).sum())

print("\n11. ROW RECONCILIATION")
print("Rows removed:", len(df_original) - len(df))
print("Expected rows after removing duplicates:", len(df_original) - validation["exact_duplicates"])
print("Actual final rows:", len(df))


1. DATASET SIZE
Original rows: 4687
Original columns: 14
Final rows: 4565

2. MISSING VALUES IN ORIGINAL DATA
user_id                0
parent_asin            0
rating                 0
review_title           0
review_text            0
timestamp              0
helpful_vote           0
verified_purchase      0
product_title          0
product_description    0
product_price          0
product_brand          0
product_categories     0
image_url              0
dtype: int64

3. EXACT DUPLICATES
Exact duplicate rows: 122

4. DATA TYPES
user_id                    str
parent_asin                str
rating                 float64
review_title               str
review_text                str
timestamp                int64
helpful_vote             int64
verified_purchase        int64
product_title              str
product_description        str
product_price          float64
product_brand              str
product_categories         str
image_url                  str
dtype: object

5. RATING CHECK

## 6. Metrics

These are the main metrics for the project question. I also compared it to the orignial df to see how cleaning the data can influence results. 

In [25]:
print("Metrics before Cleaning")
print("Average rating:", df_original["rating"].mean())
print("\nRating distribution:")
print(df_original["rating"].value_counts().sort_index())
print("\nVerified vs. unverified ratings:")
print(df_original.groupby("verified_purchase")["rating"].agg(["count", "mean"]))
print("\nAverage helpful votes:", df_original["helpful_vote"].mean())
#print("Average review length:", df_original["review_length"].mean())

print("\nMetrics after Cleaning")
print("Average rating:", df["rating"].mean())
print("\nRating distribution:")
print(df["rating"].value_counts().sort_index())
print("\nVerified vs. unverified ratings:")
print(df.groupby("verified_purchase")["rating"].agg(["count", "mean"]))
print("\nAverage helpful votes:", df["helpful_vote"].mean())
print("Average review length:", df["review_length"].mean())


Metrics before Cleaning
Average rating: 4.2154896522295715

Rating distribution:
rating
1.0     256
2.0     227
3.0     468
4.0    1036
5.0    2700
Name: count, dtype: int64

Verified vs. unverified ratings:
                   count      mean
verified_purchase                 
0                   2801  4.274902
1                   1886  4.127253

Average helpful votes: 1.0637934713036057

Metrics after Cleaning
Average rating: 4.228258488499453

Rating distribution:
rating
1.0     233
2.0     216
3.0     462
4.0    1019
5.0    2635
Name: count, dtype: int64

Verified vs. unverified ratings:
                   count      mean
verified_purchase                 
False               2791  4.279828
True                1774  4.147125

Average helpful votes: 1.0705366922234392
Average review length: 432.1934282584885


## 7. Five record checks

These checks compare an original record or cleaning decision with the expected result and the actual cleaned result.

In [22]:
checks = make_record_checks(df_original, df)
checks

,check,original,expected,actual,matches
0,1 - timestamp conversion,1627391044559,2021-07-27 13:04:04.559000,2021-07-27 13:04:04.559000,True
1,2 - verified purchase conversion,0,False,False,True
2,3 - review length calculation,351 original characters,351,351,True
3,4 - blank review text handling,Blank review text,Keep blank and review_length = 0,Blank review text and review_length = 0,True
4,5 - exact duplicate handling,122 duplicate rows in source,Remove duplicate copies and keep first occurrence,122 rows removed; 4565 rows remain,True


## 8. Save outputs


In [26]:
out = save_outputs(df_original, df)
print("Outputs saved to:", out.resolve())
print("\nOutput files:")
for file in sorted(out.iterdir()):
    print(file.name)

Outputs saved to: /Users/lamewadipeba/dso576/project/outputs

Output files:
cleaned_all_beauty_sample.csv
cleaned_sample_50_rows.csv
five_record_checks.csv
metrics.csv
validation_summary.csv
verified_purchase_comparison.csv
